# 한국어 가맹점 의미 검색 — BGE-M3 원본 기준선

이 노트북은 **모델을 학습하거나 파인튜닝하지 않습니다.** 이미 학습된 `BAAI/bge-m3`로 `가맹점명 + 원본 취급품목` 문서를 임베딩하고, Colab 안의 Qdrant 로컬 모드에서 검색합니다.

목표는 전처리 전 기준선 결과를 빠르게 확인하는 것입니다. 결과가 확인되면 동일한 질의로 정규화·NICE 분류 버전과 비교합니다.

> 회사 데이터 외부 반출 정책을 먼저 확인하세요. 원본 Excel 대신 로컬에서 생성한 `search_documents_raw.csv`만 업로드합니다. 이 CSV에도 가맹점명과 취급품목이 있으므로 승인 범위를 확인해야 합니다.

## 0. Colab 런타임 설정

Colab 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU`를 선택하세요.

In [ ]:
!pip -q install -U FlagEmbedding qdrant-client pandas


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from google.colab import files
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient, models

print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 1. 로컬에서 만든 검색 문서 CSV 업로드

프로젝트 PC에서 먼저 다음 명령을 실행합니다. 암호는 프롬프트에 입력하며 화면에 표시되지 않습니다.

```powershell
python -m src.prepare_experiment "data/raw/merchant.xlsx" --ask-password
```

생성된 `data/processed/search_documents_raw.csv`를 아래 셀에서 업로드하세요.

In [ ]:
uploaded = files.upload()
csv_candidates = [name for name in uploaded if name.lower().endswith('.csv')]
cache_candidates = [name for name in uploaded if name == 'bge_m3_raw_dense_vectors.npz']
assert len(csv_candidates) == 1, 'search_documents_raw.csv 한 개를 업로드하세요.'
csv_name = csv_candidates[0]
cache_name = cache_candidates[0] if cache_candidates else None
df = pd.read_csv(csv_name, dtype=str).fillna('')
df['record_id'] = df['record_id'].astype(int)

required = {'record_id', 'merchant_name', 'original_items', 'market_name', 'search_document'}
missing = required - set(df.columns)
assert not missing, f'필수 컬럼 누락: {missing}'
assert df['record_id'].is_unique, 'record_id가 중복되었습니다.'

print('문서 수:', len(df))
print('취급품목 결측:', df['original_items'].eq('').sum())
display(df[['record_id', 'merchant_name', 'original_items', 'search_document']].head(10))


확인할 사항:

- 품목이 있으면 `가맹점명: OO. 취급품목: 생닭.`
- 품목이 없으면 `가맹점명: OO.`
- 문자열 `NULL`은 임베딩하지 않음
- 정규화·분류·추론은 하지 않음

## 2. BGE-M3 로드

첫 실행에는 모델 다운로드 시간이 걸립니다. T4 GPU에서는 FP16을 사용합니다.

In [ ]:
MODEL_NAME = 'BAAI/bge-m3'
model = BGEM3FlagModel(
    MODEL_NAME,
    use_fp16=torch.cuda.is_available(),
)


## 3. 전체 가맹점 임베딩 생성

이번 기준선은 dense vector만 사용합니다. BGE-M3의 dense vector 크기는 1,024입니다.

In [ ]:
documents = df['search_document'].tolist()
started = time.time()

if cache_name:
    cache = np.load(cache_name)
    cached_ids = cache['record_ids'].astype(int)
    expected_ids = df['record_id'].to_numpy(dtype=int)
    assert np.array_equal(cached_ids, expected_ids), '캐시와 CSV의 record_id 순서가 다릅니다.'
    document_vectors = cache['vectors'].astype(np.float32)
    print('검증된 임베딩 캐시를 불러왔습니다.')
else:
    encoded = model.encode(
        documents,
        batch_size=32,
        max_length=128,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    document_vectors = np.asarray(encoded['dense_vecs'], dtype=np.float32)

assert document_vectors.shape == (len(df), 1024)
print('벡터 shape:', document_vectors.shape)
print(f'준비 시간: {time.time() - started:.1f}초')


메모리 부족이 발생하면 `batch_size=16` 또는 `8`로 낮추세요. 문서가 짧기 때문에 `max_length=128`이면 충분합니다. 재실행할 때는 CSV와 11번에서 받은 `.npz` 캐시를 함께 업로드하면 임베딩 생성을 건너뜁니다.

## 4. Qdrant 로컬 컬렉션 생성

`QdrantClient(':memory:')`는 별도 서버 없이 Colab 메모리에서 실행됩니다. 런타임이 종료되면 컬렉션도 사라집니다.

In [ ]:
COLLECTION_NAME = 'merchant_raw_bge_m3'
client = QdrantClient(':memory:')
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(size=1024, distance=models.Distance.COSINE),
)

UPLOAD_BATCH_SIZE = 256
for start in range(0, len(df), UPLOAD_BATCH_SIZE):
    end = min(start + UPLOAD_BATCH_SIZE, len(df))
    points = []
    for position in range(start, end):
        row = df.iloc[position]
        points.append(
            models.PointStruct(
                id=int(row['record_id']),
                vector=document_vectors[position].tolist(),
                payload={
                    'merchant_name': row['merchant_name'],
                    'original_items': row['original_items'],
                    'market_name': row['market_name'],
                    'search_document': row['search_document'],
                },
            )
        )
    client.upsert(collection_name=COLLECTION_NAME, points=points)

print(client.get_collection(COLLECTION_NAME))


## 5. 의미 검색 함수

In [ ]:
def search_merchants(query: str, limit: int = 10) -> pd.DataFrame:
    query_vector = model.encode(
        [query],
        batch_size=1,
        max_length=128,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )['dense_vecs'][0]

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=np.asarray(query_vector, dtype=np.float32).tolist(),
        limit=limit,
        with_payload=True,
    ).points

    return pd.DataFrame([
        {
            'rank': rank,
            'score': round(hit.score, 4),
            'record_id': hit.id,
            'merchant_name': hit.payload.get('merchant_name', ''),
            'original_items': hit.payload.get('original_items', ''),
            'market_name': hit.payload.get('market_name', ''),
        }
        for rank, hit in enumerate(hits, start=1)
    ])


In [ ]:
search_merchants('생닭 파는 곳', limit=10)


## 6. 원본 기준선 질의 점검

아래 질의는 결과를 눈으로 확인하기 위한 시작점입니다. 아직 정답셋이 없으므로 이 단계의 평가는 정량 성능평가가 아닙니다.

In [ ]:
test_queries = [
    '생닭 파는 곳',
    '국거리 고기 살 수 있는 가게',
    '반찬 사는 곳',
    '커피 마실 수 있는 곳',
    '아이 옷 파는 가게',
    '떡케이크 주문할 곳',
    '주방용품 파는 곳',
    '화장품 가게',
    '도마큰시장 과일가게',
]

for query in test_queries:
    print(f'\n### 질의: {query}')
    display(search_merchants(query, limit=5))


결과를 볼 때 다음을 기록하세요.

1. 상위 5개가 질의 의도에 맞는가?
2. 정확한 품목 표현이 없어도 동의어가 검색되는가?
3. `음식`, `식품` 같은 모호한 품목이 과도하게 노출되는가?
4. 취급품목 결측 가맹점은 이름만으로 검색되는가?
5. 지역·시장명 질의는 현재 문서에 시장명이 없으므로 잘 안 되는 것이 정상이다. 이후 버전에서 시장명 포함 또는 Qdrant 필터로 비교한다.

## 7. 평가 질의와 기준선 결과를 CSV로 저장

아래 25개는 1차 기준선용입니다. `query_id`는 이후 데이터 버전과 모델이 바뀌어도 유지합니다. 질의 문구와 의도가 실제 서비스에 적합한지는 사람이 검토해야 합니다.

In [ ]:
evaluation_queries = pd.DataFrame([
    ('Q001', '생닭 파는 곳', 'specific_item', '생닭 판매점', '', '생닭|닭'),
    ('Q002', '반찬 사는 곳', 'specific_item', '반찬 판매점', '', '반찬'),
    ('Q003', '커피 마실 수 있는 곳', 'specific_item', '커피 판매점 또는 카페', '', '커피|카페'),
    ('Q004', '화장품 가게', 'specific_item', '화장품 판매점', '', '화장품'),
    ('Q005', '주방용품 파는 곳', 'specific_item', '주방용품 판매점', '', '주방용품|주방기구|그릇'),
    ('Q006', '국거리 고기 살 수 있는 가게', 'colloquial', '정육 또는 육류 판매점; 음식점 제외', '', '정육|소고기|식육|육류'),
    ('Q007', '아이 옷 파는 가게', 'colloquial', '아동복 판매점', '', '아동복'),
    ('Q008', '밑반찬 살 만한 데', 'colloquial', '반찬 판매점', '', '반찬'),
    ('Q009', '얼굴 관리받는 곳', 'colloquial', '피부관리 또는 미용 서비스', '', '피부관리|미용'),
    ('Q010', '책 살 수 있는 곳', 'colloquial', '서점 또는 도서 판매점', '', '서적|도서|서점'),
    ('Q011', '먹거리 파는 곳', 'broad_category', '식품 또는 음식 관련 가맹점', '', '식품|음식'),
    ('Q012', '생활용품점', 'broad_category', '생활·가정용품 판매점', '', '생활용품|가정용품'),
    ('Q013', '축산물 가게', 'broad_category', '축산물 판매점', '', '축산물|정육|식육'),
    ('Q014', '옷 가게', 'broad_category', '의류 판매점', '', '의류|옷'),
    ('Q015', '미용 관련 가게', 'broad_category', '미용실 또는 미용 서비스', '', '미용|미용실'),
    ('Q016', '도마큰시장 과일가게', 'market_item', '도마큰시장 내 과일 판매점', '도마큰시장', '과일|청과'),
    ('Q017', '한민시장 떡집', 'market_item', '한민시장 내 떡 판매점', '한민시장', '떡'),
    ('Q018', '중리시장 건어물 가게', 'market_item', '중리시장 내 건어물 판매점', '중리시장', '건어물'),
    ('Q019', '문창시장 반찬가게', 'market_item', '문창시장 내 반찬 판매점', '문창시장', '반찬'),
    ('Q020', '유성시장 주방용품점', 'market_item', '유성시장 골목형상점가 내 주방용품 판매점', '유성시장 골목형상점가', '주방용품|주방기구'),
    ('Q021', '수정반찬 찾기', 'missing_item_name', '취급품목 결측 가맹점명 검색', '', '반찬'),
    ('Q022', '커피 그림', 'missing_item_name', '취급품목 결측 가맹점명 검색', '', '커피'),
    ('Q023', '피부이야기', 'missing_item_name', '취급품목 결측 가맹점명 검색', '', '피부|화장품'),
    ('Q024', '미젤 화장품', 'missing_item_name', '취급품목 결측 가맹점명 검색', '', '화장품'),
    ('Q025', '떡방 고구려', 'missing_item_name', '취급품목 결측 가맹점명 검색', '', '떡'),
], columns=['query_id', 'query', 'query_type', 'intent', 'must_market', 'expected_items'])

evaluation_queries['review_status'] = 'draft'
evaluation_queries.to_csv('evaluation_queries_v1.csv', index=False, encoding='utf-8-sig')
display(evaluation_queries)
# 질의 검토용 파일이 필요하면 다음 줄의 주석을 해제하세요.
# files.download('evaluation_queries_v1.csv')


In [ ]:
EXPERIMENT_ID = 'raw_bge_m3_v1'
DATA_VERSION = 'raw_name_and_original_items'
TOP_K = 5

result_frames = []
for item in evaluation_queries.itertuples(index=False):
    result = search_merchants(item.query, limit=TOP_K)
    result.insert(0, 'query_type', item.query_type)
    result.insert(0, 'query', item.query)
    result.insert(0, 'query_id', item.query_id)
    result.insert(0, 'data_version', DATA_VERSION)
    result.insert(0, 'model', MODEL_NAME)
    result.insert(0, 'experiment_id', EXPERIMENT_ID)
    result_frames.append(result)

baseline_results = pd.concat(result_frames, ignore_index=True)
baseline_results['relevance'] = ''
baseline_results['review_note'] = ''
baseline_results.to_csv('baseline_results_for_review.csv', index=False, encoding='utf-8-sig')

print('검토할 행 수:', len(baseline_results))
display(baseline_results.head(20))
files.download('baseline_results_for_review.csv')


다운로드한 `baseline_results_for_review.csv`에서 사람이 두 컬럼만 작성합니다.

- `relevance=2`: 질의 의도를 명확하게 만족
- `relevance=1`: 부분 관련 또는 실제 업종 확인 필요
- `relevance=0`: 무관
- `review_note`: 0점·1점 또는 애매한 사례에만 이유 작성

모델의 `score`는 정답 확률이 아니므로 관련도 판정 기준으로 사용하지 않습니다.

## 8. 사람이 판정한 CSV로 기준선 지표 계산

Excel에서 `relevance`를 모두 입력한 파일을 다시 업로드합니다. 현재 정답 풀은 검색 후보만 검토하므로 완전한 Recall 대신 Hit@5, Precision@5, MRR@5, nDCG@5를 계산합니다.

In [ ]:
labeled_upload = files.upload()
labeled_name = next(iter(labeled_upload))
labeled = pd.read_csv(labeled_name)
labeled['relevance'] = pd.to_numeric(labeled['relevance'], errors='coerce')

if labeled['relevance'].isna().any():
    missing_count = int(labeled['relevance'].isna().sum())
    raise ValueError(f'relevance가 비어 있거나 숫자가 아닌 행이 {missing_count}개 있습니다.')
if not labeled['relevance'].isin([0, 1, 2]).all():
    raise ValueError('relevance는 0, 1, 2 중 하나여야 합니다.')

def evaluate_group(group: pd.DataFrame) -> pd.Series:
    ranked = group.sort_values('rank')
    top5 = ranked.head(5)
    binary5 = top5['relevance'].eq(2)
    first_positions = np.flatnonzero(binary5.to_numpy())
    mrr5 = 0.0 if len(first_positions) == 0 else 1.0 / (first_positions[0] + 1)

    gains = (2 ** top5['relevance'].to_numpy(dtype=float) - 1)
    discounts = np.log2(np.arange(2, len(top5) + 2))
    dcg = float(np.sum(gains / discounts))
    ideal = np.sort(top5['relevance'].to_numpy(dtype=float))[::-1]
    idcg = float(np.sum((2 ** ideal - 1) / discounts))

    return pd.Series({
        'hit_at_5': float(binary5.any()),
        'precision_at_5': float(binary5.mean()),
        'mrr_at_5': mrr5,
        'ndcg_at_5': 0.0 if idcg == 0 else dcg / idcg,
    })

query_metrics = labeled.groupby(['experiment_id', 'query_id', 'query', 'query_type']).apply(
    evaluate_group, include_groups=False
).reset_index()
summary_metrics = query_metrics.groupby('experiment_id')[
    ['hit_at_5', 'precision_at_5', 'mrr_at_5', 'ndcg_at_5']
].mean().reset_index()

display(summary_metrics)
display(query_metrics.sort_values(['hit_at_5', 'ndcg_at_5']).head(10))
query_metrics.to_csv('query_metrics_raw_bge_m3_v1.csv', index=False, encoding='utf-8-sig')
summary_metrics.to_csv('summary_metrics_raw_bge_m3_v1.csv', index=False, encoding='utf-8-sig')
files.download('query_metrics_raw_bge_m3_v1.csv')
files.download('summary_metrics_raw_bge_m3_v1.csv')


## 9. 시장명은 임베딩 대신 Qdrant 필터로 비교

`도마큰시장 과일가게`처럼 정확한 시장 조건이 있는 질의는 시장을 필터로 제한하고 남은 품목 표현만 임베딩합니다. 이것은 데이터 정규화와 별개의 검색 파이프라인 개선입니다.

In [ ]:
def search_merchants_with_market(
    semantic_query: str,
    market_name: str,
    limit: int = 10,
) -> pd.DataFrame:
    query_vector = model.encode(
        [semantic_query],
        batch_size=1,
        max_length=128,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )['dense_vecs'][0]

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=np.asarray(query_vector, dtype=np.float32).tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key='market_name',
                    match=models.MatchValue(value=market_name),
                )
            ]
        ),
        limit=limit,
        with_payload=True,
    ).points

    return pd.DataFrame([
        {
            'rank': rank,
            'score': round(hit.score, 4),
            'record_id': hit.id,
            'merchant_name': hit.payload.get('merchant_name', ''),
            'original_items': hit.payload.get('original_items', ''),
            'market_name': hit.payload.get('market_name', ''),
        }
        for rank, hit in enumerate(hits, start=1)
    ])


In [ ]:
print('필터 없음: 도마큰시장 과일가게')
display(search_merchants('도마큰시장 과일가게', limit=5))

print('시장 필터 적용: 도마큰시장 + 과일가게')
display(search_merchants_with_market('과일가게', '도마큰시장', limit=5))


평가 CSV는 웹 검색 때 직접 조회하는 데이터가 아닙니다. 모델·문서 구성·필터 변경 후 품질이 나빠지지 않았는지 반복 검증하는 회귀 테스트 자산입니다. 실제 웹 검색은 `검색창 → 검색 API → 질의 임베딩 및 조건 필터 → Qdrant → 가맹점 상세 조회 → 결과 표시` 순서로 동작합니다.

## 10. 시연용 검색창

검색어를 입력하고 필요하면 시장명을 선택합니다. 시장명을 선택하면 Qdrant의 정확 필터를 함께 적용합니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

query_input = widgets.Text(
    value='생닭 파는 곳',
    description='검색어:',
    layout=widgets.Layout(width='650px'),
)
market_options = ['전체 시장'] + sorted(
    market for market in df['market_name'].unique().tolist() if market
)
market_dropdown = widgets.Dropdown(
    options=market_options,
    value='전체 시장',
    description='시장:',
    layout=widgets.Layout(width='650px'),
)
search_button = widgets.Button(description='검색', button_style='primary')
search_output = widgets.Output()

def run_demo_search(_):
    with search_output:
        clear_output(wait=True)
        query = query_input.value.strip()
        if not query:
            print('검색어를 입력하세요.')
            return
        if market_dropdown.value == '전체 시장':
            result = search_merchants(query, limit=5)
        else:
            result = search_merchants_with_market(
                query, market_dropdown.value, limit=5
            )
        display(result)

search_button.on_click(run_demo_search)
display(widgets.VBox([query_input, market_dropdown, search_button, search_output]))


## 11. 임베딩 캐시 다운로드(선택)

다음 실행에서 모델 임베딩 시간을 줄이려면 벡터와 문서 CSV를 함께 보관할 수 있습니다. 회사 데이터 보관 정책을 확인하세요.

In [ ]:
np.savez_compressed(
    'bge_m3_raw_dense_vectors.npz',
    vectors=document_vectors,
    record_ids=df['record_id'].to_numpy(dtype=int),
)
# 필요할 때만 주석을 해제하세요.
# files.download('bge_m3_raw_dense_vectors.npz')


## 다음 단계

이 기준선 결과를 저장한 뒤 동일한 질의로 다음 버전을 비교합니다.

- 가맹점명만
- 가맹점명 + 원본 취급품목(현재 버전)
- 취급품목 결측 보완
- 원본 취급품목 + NICE 대·중·소분류
- BGE-M3와 한국어 특화 임베딩 모델 비교

파인튜닝은 `검색어 → 정답 가맹점` 평가·학습 쌍을 확보한 이후에 진행합니다.